In [ ]:
#@title Install
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ms-swift==4.4.2
!pip install -q "transformers==4.57.3" "datasets==3.6.0" "torchao>=0.16.0"
!pip install -q qwen-omni-utils soundfile decord scipy matplotlib tqdm h5py accelerate
import importlib.metadata as _md
for p in ('ms-swift','transformers','datasets','torchao','numpy','scipy'):
    try: print(f'  {p:14}{_md.version(p)}')
    except Exception: print(f'  {p:14}not installed')
print()
print('restart the runtime, then run the next cell')

In [ ]:
#@title Configuration
import os, json, re, math, random, glob, time, sys
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict

ROOT = Path('/content/drive/MyDrive/bearing_protocol')
PROJECT = ROOT
RAW=ROOT/'cwru_raw'; DATA_DIR=ROOT/'data_cw_5khz'
JSONL_DIR=ROOT/'jsonl'; CKPT_ROOT=ROOT/'ckpt'; RUNS_DIR=ROOT/'runs'
EVAL_DIR=ROOT/'eval'; SUBSET_DIR=EVAL_DIR/'subsets'; MODEL_CACHE=ROOT/'model_cache'
for p in (DATA_DIR, DATA_DIR/'train', DATA_DIR/'test', JSONL_DIR, CKPT_ROOT,
          RUNS_DIR, EVAL_DIR, SUBSET_DIR): p.mkdir(parents=True, exist_ok=True)
os.environ['MODELSCOPE_CACHE']=str(MODEL_CACHE)

DATASET='CW'; MODEL_TAG='qwen3omni-30b'; MODALITY='both'
MODEL_ID='Qwen/Qwen3-Omni-30B-A3B-Instruct'; MODEL_TYPE='qwen3_omni_moe'
CONTRASTIVE_RATIO = 60
SEED              = 0
BLIND_RATIO=0.05
assert CONTRASTIVE_RATIO in (0, 30, 45, 60, 75, 100), 'not a ratio of the sweep'
SPEC_ID='cw_5khz_inferno_pg03_v2'
TEMPLATE_ID='v22_measured_evidence_v2'

SPLIT_AXIS='load'
HOLDOUT_DIAMETER=0.014
HEALTHY_TEST_LOAD=3
OUTER_ORIENTATIONS='all'

BALANCE_TRAIN=True
BALANCE_MODE='oversample'
VARIANT=(('split'+SPLIT_AXIS)
         + (('bal'+BALANCE_MODE) if BALANCE_TRAIN else '')
         + ('' if OUTER_ORIENTATIONS=='all' else OUTER_ORIENTATIONS))
STFT_MAX_FREQ_LABEL=5000

RUN_TAG=f'{DATASET}_{MODEL_TAG}_{MODALITY}_r{CONTRASTIVE_RATIO}_s{SEED}'

NUM_EPOCHS=4; BATCH_SIZE=1; GRAD_ACCUM=4; LR=1e-4
LORA_RANK=16; LORA_ALPHA=64; MAX_LENGTH=4096
SAVE_STEPS=100; SAVE_LIMIT=2

SEGMENT_DURATION=2.0; OVERLAP_RATIO=0.5; MAX_SEGMENTS_PER_FILE=20
VIB_SR=12000; TARGET_SR=16000
STFT_NPERSEG=384
STFT_MAX_FREQ=5000.0; STFT_IMG_W, STFT_IMG_H = 640, 480
STFT_CMAP='inferno'; STFT_GAMMA=0.3
BANDPASS_LOW, BANDPASS_HIGH = 1000.0, 10000.0; FREQ_BAND_WIDTH=5.0
assert abs(VIB_SR/STFT_NPERSEG - 64000/2048) < 1e-9, 'bin width differs from PB'

N_BALLS, D_BALL, D_PITCH = 9, 0.3126, 1.537
_r=D_BALL/D_PITCH
BPFO_COEFF=(N_BALLS/2)*(1-_r); BPFI_COEFF=(N_BALLS/2)*(1+_r)
CLASSES=['Healthy','Inner_Ring_Fault','Outer_Ring_Fault']
random.seed(SEED); np.random.seed(SEED)

def rec_path(rid): return RUNS_DIR/f'{rid}.json'
def rec_load(rid):
    p=rec_path(rid)
    return json.load(open(p)) if p.exists() else None
def rec_update(rid, **kw):
    r=rec_load(rid) or {'run_id':rid}
    r.update(kw); rec_path(rid).write_text(json.dumps(r, indent=1, default=str))
    return r

print('RUN_TAG  ', RUN_TAG)
print(f'bin width {VIB_SR/STFT_NPERSEG:.2f} Hz, '
      f'{int(STFT_MAX_FREQ/(VIB_SR/STFT_NPERSEG))} bins to 5 kHz')
print(f'BPFO {BPFO_COEFF:.4f} f_r   BPFI {BPFI_COEFF:.4f} f_r')
print(f'balance  {BALANCE_MODE if BALANCE_TRAIN else "OFF"}   epochs {NUM_EPOCHS}')
print(f'split    {SPLIT_AXIS}'
      + (f' (hold out {HOLDOUT_DIAMETER}\")' if SPLIT_AXIS=='severity'
         else f' (train 0/1/2 hp, test {HEALTHY_TEST_LOAD} hp)'))
_out=CKPT_ROOT/f'{RUN_TAG}_{TEMPLATE_ID}'
_ck=[p for p in glob.glob(str(_out/'**'/'checkpoint-*'), recursive=True) if os.path.isdir(p)]
print(f'{_out}: {len(_ck)} checkpoint(s), training {"resumes" if _ck else "starts from scratch"}')

In [ ]:
#@title Signal processing
import scipy.io, soundfile as sf, h5py
from math import gcd
from tqdm import tqdm
from scipy import stats as scipy_stats
from scipy.signal import (butter, filtfilt, hilbert, resample_poly,
                          stft as scipy_stft)
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm

def read_mat(p):
    try:
        return scipy.io.loadmat(str(p))
    except Exception:
        with h5py.File(p,'r') as f:
            return {k: np.array(f[k]).squeeze() for k in f.keys()}

def de_signal(mat):
    ks=[k for k in mat if isinstance(k,str) and k.endswith('DE_time')]
    return np.ravel(mat[ks[0]]).astype(np.float64) if ks else None

def rpm_of(mat):
    for k in mat:
        if isinstance(k,str) and k.endswith('RPM'):
            try: return float(np.ravel(mat[k])[0])
            except Exception: pass
    return None

def vibration_to_audio(signal, source_sr, target_sr=TARGET_SR):
    signal = signal - np.mean(signal)
    g = gcd(int(source_sr), int(target_sr))
    rs = resample_poly(signal, int(target_sr//g), int(source_sr//g))
    m = np.max(np.abs(rs))
    if m > 0: rs = rs/m*0.95
    return rs.astype(np.float32)

def gen_stft_image(vib_segment, vib_sr, output_path, max_freq=None):
    max_freq = STFT_MAX_FREQ if max_freq is None else max_freq
    f, t, Zxx = scipy_stft(vib_segment, fs=vib_sr, nperseg=STFT_NPERSEG,
                           noverlap=STFT_NPERSEG//2)
    m = f <= max_freq
    f = f[m]; Zxx = Zxx[m, :]
    fig, ax = plt.subplots(1,1, figsize=(STFT_IMG_W/100, STFT_IMG_H/100), dpi=100)
    ax.pcolormesh(t, f, np.abs(Zxx), shading='gouraud', cmap=STFT_CMAP,
                  norm=PowerNorm(gamma=STFT_GAMMA))
    ax.set_ylim(0, max_freq); ax.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

def bandpass_filter(sig, lo, hi, sr, order=5):
    nyq=0.5*sr
    b,a=butter(order,[lo/nyq, min(hi/nyq, 0.99)], btype='band')
    return filtfilt(b,a,sig)

def envelope_spectrum(sig, sr, lo, hi):
    env=np.abs(hilbert(bandpass_filter(sig,lo,hi,sr)))
    env=env-np.mean(env)
    return np.fft.rfftfreq(len(env), d=1.0/sr), np.abs(np.fft.rfft(env))

def extract_features_adaptive(vib_segment, vib_sr, speed_rpm):
    rms=float(np.sqrt(np.mean(vib_segment**2)))
    kurtosis=float(scipy_stats.kurtosis(vib_segment, fisher=True))
    peak=float(np.max(np.abs(vib_segment)))
    crest_factor=float(peak/rms) if rms>0 else 0.0
    f_shaft=speed_rpm/60.0
    bpfo_hz=BPFO_COEFF*f_shaft; bpfi_hz=BPFI_COEFF*f_shaft
    try:
        ef, em = envelope_spectrum(vib_segment, vib_sr, BANDPASS_LOW, BANDPASS_HIGH)
        def be(c, bw=FREQ_BAND_WIDTH):
            m=(ef>=c-bw)&(ef<=c+bw)
            return float(np.sum(em[m]**2)) if np.any(m) else 0.0
        bpfo_1x,bpfo_2x,bpfo_3x = be(bpfo_hz), be(2*bpfo_hz), be(3*bpfo_hz)
        bpfi_1x,bpfi_2x         = be(bpfi_hz), be(2*bpfi_hz)
        side                    = be(bpfi_hz-f_shaft)
        fault_band=(ef>=20)&(ef<=500)
        total=max(float(np.sum(em[fault_band]**2)),1e-10)
        bpfo_ratio=(bpfo_1x+bpfo_2x)/total
        bpfi_ratio=(bpfi_1x+bpfi_2x)/total
        harm=(bpfo_2x+bpfo_3x)/total
        side_r=side/total
        char=(bpfo_1x+bpfo_2x+bpfo_3x+bpfi_1x+bpfi_2x+side)/total
        min_bin=max(1,int(20*len(ef)/(vib_sr/2)))
        max_bin=int(500*len(ef)/(vib_sr/2))
        rng=em[min_bin:max_bin]
        dom=float(ef[np.argmax(rng)+min_bin]) if len(rng)>0 else 0.0
    except Exception:
        bpfo_ratio=bpfi_ratio=harm=side_r=char=0.0; dom=0.0
    return {'rms':round(rms,4),'kurtosis':round(kurtosis,2),
            'crest_factor':round(crest_factor,2),'speed_rpm':round(speed_rpm,1),
            'f_shaft_hz':round(f_shaft,2),'bpfo_hz':round(bpfo_hz,1),
            'bpfi_hz':round(bpfi_hz,1),'env_dominant_freq_hz':round(dom,1),
            'env_bpfo_ratio':round(bpfo_ratio,4),'env_bpfi_ratio':round(bpfi_ratio,4),
            'env_bpfo_harmonic_ratio':round(harm,4),
            'env_bpfi_sideband_ratio':round(side_r,4),
            'env_char_energy_frac':round(char,4)}
_g=gcd(VIB_SR, TARGET_SR)
print(f'audio resample {VIB_SR} -> {TARGET_SR}: up {TARGET_SR//_g}, down {VIB_SR//_g}')

In [ ]:
#@title Download the CWRU recordings
import urllib.request, urllib.error, shutil, tempfile

TABLE_12K={
 ('Healthy',          None,  None): [ 97,  98,  99, 100],
 ('Inner_Ring_Fault', 0.007, None): [105, 106, 107, 108],
 ('Inner_Ring_Fault', 0.014, None): [169, 170, 171, 172],
 ('Inner_Ring_Fault', 0.021, None): [209, 210, 211, 212],
 ('Outer_Ring_Fault', 0.007, '@6' ): [130, 131, 132, 133],
 ('Outer_Ring_Fault', 0.007, '@3' ): [144, 145, 146, 147],
 ('Outer_Ring_Fault', 0.007, '@12'): [156, 158, 159, 160],
 ('Outer_Ring_Fault', 0.014, '@6' ): [197, 198, 199, 200],
 ('Outer_Ring_Fault', 0.021, '@6' ): [234, 235, 236, 237],
 ('Outer_Ring_Fault', 0.021, '@3' ): [246, 247, 248, 249],
 ('Outer_Ring_Fault', 0.021, '@12'): [258, 259, 260, 261],
}
EXCLUDE=set(range(109,113))|set(range(122,126))|set(range(135,139)) \
       |set(range(148,152))|set(range(161,165))|set(range(174,178)) \
       |set(range(189,193))|set(range(201,205))|set(range(213,218)) \
       |set(range(226,230))|set(range(238,242))|set(range(250,254)) \
       |set(range(262,266))
NUM2META={n:{'class':c_,'diameter':d,'position':p,'load':l}
          for (c_,d,p),nums in TABLE_12K.items() for l,n in enumerate(nums)}
EXPECTED_RPM={0:1797,1:1772,2:1750,3:1730}

WANTED={n:f"{c_} {d or ''} {p or ''} {l} hp".replace('  ',' ')
        for (c_,d,p),nums in TABLE_12K.items() if c_ in CLASSES for l,n in enumerate(nums)}
URL_PATTERNS = [
 'https://engineering.case.edu/sites/default/files/{n}.mat',
 'https://engineering.case.edu/sites/default/files/{n}.mat?download=1',
 'https://csegroups.case.edu/sites/default/files/bearingdatacenter/files/Datafiles/{n}.mat',
 'https://raw.githubusercontent.com/s-whynot/CWRU-dataset/master/12DriveEndFault/{n}.mat',
 'https://huggingface.co/datasets/mtdt/CWRU/resolve/main/{n}.mat',
]

def valid_mat(path):
    try:
        m = read_mat(path)
        return de_signal(m) is not None
    except Exception:
        return False

RAW.mkdir(parents=True, exist_ok=True)
got, failed = [], []
for num, label in sorted(WANTED.items()):
    dest = RAW/f'{num}.mat'
    if dest.exists() and valid_mat(dest):
        got.append(num); continue
    ok = False
    for pat in URL_PATTERNS:
        url = pat.format(n=num)
        try:
            with tempfile.NamedTemporaryFile(suffix='.mat', delete=False) as tmp:
                req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=60) as r:
                    shutil.copyfileobj(r, tmp)
                tmp_path = tmp.name
            if valid_mat(tmp_path):
                shutil.move(tmp_path, dest)
                got.append(num); ok = True
                print(f'  {num:>4} {label:28} downloaded from {pat.split("/")[2]}')
                break
            os.unlink(tmp_path)
        except Exception:
            continue
    if not ok:
        failed.append(num); print(f'  {num:>4} {label:28} NOT FOUND on any mirror')
print(f'\n{len(got)} of {len(WANTED)} present, {len(failed)} missing')
if failed:
    print(f'download the missing files from https://engineering.case.edu/bearingdatacenter/'
          f'12k-drive-end-bearing-fault-data into {RAW} as <number>.mat')

In [ ]:
#@title File index
INDEX=[];excluded=[];unmapped=[];unreadable=[]
for p in sorted(RAW.rglob('*.mat')):
    m=re.match(r'^(\d+)',p.stem); num=int(m.group(1)) if m else None
    if num in EXCLUDE: excluded.append(p.stem); continue
    if num not in NUM2META: unmapped.append(p.stem); continue
    try: mat=read_mat(p); sig=de_signal(mat)
    except Exception as e: unreadable.append((p.stem,str(e)[:50])); continue
    if sig is None: unreadable.append((p.stem,'no DE channel')); continue
    INDEX.append({'path':str(p),'name':p.stem,'num':num,**NUM2META[num],
                  'rpm':rpm_of(mat),'n':len(sig)})

seg_n=int(SEGMENT_DURATION*VIB_SR); hop=int(seg_n*(1-OVERLAP_RATIO))
print(f'{len(INDEX)} mapped | {len(excluded)} excluded (48 kHz) | '
      f'{len(unmapped)} unmapped | {len(unreadable)} unreadable\n')
byc=Counter((r['class'],r['diameter'],r['position']) for r in INDEX)
for k,v in sorted(byc.items(), key=lambda x:(str(x[0][0]),str(x[0][1]),str(x[0][2]))):
    print(f'  {str(k[0]):20}{str(k[1]):>7} {str(k[2] or ""):>4}  {v} files')
if unmapped: print(f'\nunmapped: {unmapped}')
if unreadable: print(f'\nunreadable: {unreadable}')

have={r['num'] for r in INDEX}
missing=[(k,[n for n in nums if n not in have and n not in EXCLUDE])
         for k,nums in TABLE_12K.items()]
missing=[(k,v) for k,v in missing if v]
if missing:
    print('\nmissing recordings:')
    for (cls,dia,pos),nums in missing:
        print(f'    {cls:20}{str(dia):>7} {str(pos or ""):>4}  {nums}')
    n_lost=sum(len(v) for k,v in missing if k[0] in CLASSES)
    print(f'{n_lost} recordings missing')
else:
    print('\nall recordings present')

In [ ]:
#@title Split and media
usable=[r for r in INDEX if r['class'] in CLASSES]
if OUTER_ORIENTATIONS=='at6':
    usable=[r for r in usable
            if r['class']!='Outer_Ring_Fault' or r['position']=='@6']

for r in usable:
    if SPLIT_AXIS=='load':
        r['split']='test' if r['load']==HEALTHY_TEST_LOAD else 'train'
    else:
        r['split']=('test' if r['load']==HEALTHY_TEST_LOAD else 'train') \
                   if r['class']=='Healthy' else \
                   ('test' if r['diameter']==HOLDOUT_DIAMETER else 'train')
    r['bearing']=('Healthy' if r['class']=='Healthy'
                  else f"{r['class']}_{r['diameter']}{r['position'] or ''}")

print(f"{'split':7}{'class':20}{'files':>6}{'est seg':>9}   bearings")
for sp in ('train','test'):
    for cls in CLASSES:
        rows=[r for r in usable if r['split']==sp and r['class']==cls]
        est=sum(min(MAX_SEGMENTS_PER_FILE,(r['n']-seg_n)//hop+1) for r in rows)
        b=sorted({r['bearing'] for r in rows})
        print(f'{sp:7}{cls:20}{len(rows):>6}{est:>9}   {len(b)} distinct')

trf={r['path'] for r in usable if r['split']=='train'}
tef={r['path'] for r in usable if r['split']=='test'}
assert not (trf & tef), 'a recording appears in both splits'
print(f'\nrecordings in both splits: {len(trf & tef)}')

if SPLIT_AXIS=='severity':
    trb={r['bearing'] for r in usable if r['split']=='train' and r['class']!='Healthy'}
    teb={r['bearing'] for r in usable if r['split']=='test'  and r['class']!='Healthy'}
    htr={r['load'] for r in usable if r['class']=='Healthy' and r['split']=='train'}
    hte={r['load'] for r in usable if r['class']=='Healthy' and r['split']=='test'}
    assert not (trb & teb), f'bearing in both splits: {trb & teb}'
    assert not (htr & hte), 'healthy load in both splits'
    print('bearing-disjoint for the fault classes; healthy held out by load')
else:
    ltr={r['load'] for r in usable if r['split']=='train'}
    lte={r['load'] for r in usable if r['split']=='test'}
    assert not (ltr & lte), 'a load appears in both splits'
    print(f'load-disjoint: train {sorted(ltr)} hp, test {sorted(lte)} hp')

all_train, all_test = [], []
def process(rec, target_dir, out_list):
    mat=read_mat(rec['path']); vib=de_signal(mat)
    rpm=rec['rpm'] or EXPECTED_RPM[rec['load']]
    n_seg=min(MAX_SEGMENTS_PER_FILE,(len(vib)-seg_n)//hop+1)
    for i in range(n_seg):
        seg=vib[i*hop:i*hop+seg_n]
        if len(seg)<seg_n: continue
        stem=f"{rec['bearing']}_{rec['name']}_L{rec['load']}_{i:03d}"
        ap=target_dir/f'{stem}.wav'; sp_=target_dir/f'{stem}_stft.png'
        if not ap.exists():
            sf.write(str(ap), vibration_to_audio(seg, VIB_SR, TARGET_SR), TARGET_SR)
        if not sp_.exists(): gen_stft_image(seg, VIB_SR, sp_)
        out_list.append({'audio':ap.name,'image':str(sp_),'label':rec['class'],
                         'bearing':rec['bearing'],'source':rec['name'],
                         'load':rec['load'],'diameter':rec['diameter'],
                         'position':rec['position'],
                         'features':extract_features_adaptive(seg, VIB_SR, rpm)})

for r in tqdm([x for x in usable if x['split']=='train'], desc='train'):
    process(r, DATA_DIR/'train', all_train)
for r in tqdm([x for x in usable if x['split']=='test'], desc='test'):
    process(r, DATA_DIR/'test', all_test)
print(f"\ntrain {len(all_train)}  {dict(Counter(s['label'] for s in all_train))}")
print(f"test  {len(all_test)}  {dict(Counter(s['label'] for s in all_test))}")
manifest={'train':all_train,'test':all_test}
_mp=DATA_DIR/'data_manifest_cw_v2.json'
if _mp.exists():
    _old=json.load(open(_mp))
    _sig=lambda m: {sp: sorted((r['audio'], r['label'], json.dumps(r['features'], sort_keys=True)) for r in m[sp])
                    for sp in ('train','test')}
    assert _sig(_old)==_sig(manifest), 'data_manifest_cw_v2.json differs from the data built here'
    print('manifest matches the one on disk')
else:
    json.dump(manifest, open(_mp,'w')); print('manifest written:', _mp.name)

In [ ]:
#@title Thresholds
def sep(feat,target,rows):
    pos=np.array([r['features'][feat] for r in rows if r['label']==target])
    neg=np.array([r['features'][feat] for r in rows if r['label']!=target])
    if len(pos)==0 or len(neg)==0: return float('nan')
    x=np.concatenate([pos,neg]); o=x.argsort(kind='mergesort')
    ranks=np.empty(len(x)); ranks[o]=np.arange(1,len(x)+1)
    _,inv,cnt=np.unique(x,return_inverse=True,return_counts=True)
    s=np.zeros(len(cnt)); np.add.at(s,inv,ranks); ranks=(s/cnt)[inv]
    n1,n2=len(pos),len(neg)
    a=(ranks[:n1].sum()-n1*(n1+1)/2)/(n1*n2)
    return max(a,1-a)

TARGETS={'env_bpfi_ratio':'Inner_Ring_Fault','env_bpfo_ratio':'Outer_Ring_Fault',
         'env_bpfo_harmonic_ratio':'Outer_Ring_Fault',
         'env_char_energy_frac':'Healthy',
         'env_bpfi_sideband_ratio':'Inner_Ring_Fault'}
print(f"{'quantity':28}{'separability':>13}")
for q,t in TARGETS.items():
    print(f'{q:28}{sep(q,t,all_train):13.3f}')

thresholds={q:[float(np.quantile([r['features'][q] for r in all_train],1/3)),
               float(np.quantile([r['features'][q] for r in all_train],2/3))]
            for q in TARGETS}

print(f"\n{'quantity':28}{'train low/mid/high':>22}{'test low/mid/high':>22}")
for q,(lo,hi) in thresholds.items():
    def dist(rows):
        v=np.array([r['features'][q] for r in rows])
        return (100*np.mean(v<lo), 100*np.mean((v>=lo)&(v<hi)), 100*np.mean(v>=hi))
    a=dist(all_train); b=dist(all_test)
    print(f'{q:28}{a[0]:6.0f}{a[1]:6.0f}{a[2]:6.0f}   {b[0]:9.0f}{b[1]:6.0f}{b[2]:6.0f}')
print()
spec={'spec_id':SPEC_ID,'dataset':'CWRU','vib_sr_hz':VIB_SR,
      'stft_nperseg':STFT_NPERSEG,'bin_hz':VIB_SR/STFT_NPERSEG,
      'bpfo_coeff':BPFO_COEFF,'bpfi_coeff':BPFI_COEFF,
      'holdout_diameter':HOLDOUT_DIAMETER,'healthy_test_load':HEALTHY_TEST_LOAD,
      'orientations_included':True,'balanced':BALANCE_TRAIN,
      'n_train':len(all_train),'n_test':len(all_test),
      'thresholds':thresholds,
      'separability':{q:sep(q,t,all_train) for q,t in TARGETS.items()}}
_sp=DATA_DIR/'spec_cw_v2.json'
if _sp.exists():
    _old=json.load(open(_sp))['thresholds']
    for q,(lo,hi) in thresholds.items():
        assert abs(_old[q][0]-lo)<1e-9 and abs(_old[q][1]-hi)<1e-9, \
            f'{q}: thresholds {lo:.6g}/{hi:.6g} differ from spec_cw_v2.json {_old[q]}'
    print('thresholds match spec_cw_v2.json')
else:
    json.dump(spec, open(_sp,'w'), indent=2); print('spec ->', _sp)

In [ ]:
#@title Prompts and targets
import json, random
from pathlib import Path
from collections import Counter

SPEC = json.load(open(DATA_DIR/'spec_cw_v2.json'))
TH   = SPEC['thresholds']
CLASSES=['Healthy','Inner_Ring_Fault','Outer_Ring_Fault']
PRETTY={'Healthy':'healthy bearing','Inner_Ring_Fault':'inner ring fault',
        'Outer_Ring_Fault':'outer ring fault'}
ARTICLED={'Healthy':'a healthy bearing','Inner_Ring_Fault':'an inner ring fault',
          'Outer_Ring_Fault':'an outer ring fault'}

SYSTEM_PROMPT = (
 "You are an expert bearing vibration analyst. You are given an audio rendering "
 "of a bearing vibration signal and its STFT spectrogram (0-5000 Hz). A "
 "convolutional classifier may also supply a preliminary diagnosis. Examine the "
 "signal evidence and explain what it shows, stating whether it supports or "
 "contradicts any supplied diagnosis.")

def _bin(feature, value):
    lo, hi = TH[feature]
    return 'low' if value < lo else ('high' if value >= hi else 'mid')

CHAR = {'low' :"Only a small fraction of the envelope energy in the 20-500 Hz band "
                "falls at the bearing characteristic frequencies; most of it is broadband.",
        'mid' :"A moderate fraction of the envelope energy in the 20-500 Hz band falls "
                "at the bearing characteristic frequencies.",
        'high':"A large fraction of the envelope energy in the 20-500 Hz band is "
                "concentrated at the bearing characteristic frequencies."}
def OUTER_E(b,hz):
    return {'low' :f"There is little energy at the outer-race passing rate near {hz:.1f} Hz.",
            'mid' :f"There is moderate energy at the outer-race passing rate near {hz:.1f} Hz.",
            'high':f"Envelope energy is strongly concentrated at the outer-race passing "
                   f"rate near {hz:.1f} Hz."}[b]
def OUTER_H(b):
    return {'low' :"No harmonic banding is visible at twice and three times that rate.",
            'mid' :"Weak harmonic content is present at twice and three times that rate.",
            'high':"Clear harmonic banding is present at twice and three times "
                   "that rate."}[b]
def INNER_E(b,hz):
    return {'low' :f"There is little energy at the inner-race passing rate near {hz:.1f} Hz.",
            'mid' :f"There is moderate energy at the inner-race passing rate near {hz:.1f} Hz.",
            'high':f"Envelope energy is strongly concentrated at the inner-race passing "
                   f"rate near {hz:.1f} Hz."}[b]
def INNER_S(b,fs):
    return {'low' :"No shaft-rate sideband is present below that rate.",
            'mid' :"A weak shaft-rate sideband is present below that rate.",
            'high':f"A clear shaft-rate sideband appears below that rate, spaced by "
                   f"the shaft frequency of {fs:.2f} Hz."}[b]

def dominant_peak_sentence(f):
    dom, bpfo, bpfi = f['env_dominant_freq_hz'], f['bpfo_hz'], f['bpfi_hz']
    tol = max(0.08*max(bpfo,1e-6), 3.0)
    if abs(dom-bpfo) <= tol or abs(dom-2*bpfo) <= 2*tol:
        return f"The dominant envelope peak at {dom:.0f} Hz coincides with the outer-race rate."
    if abs(dom-bpfi) <= tol or abs(dom-2*bpfi) <= 2*tol:
        return f"The dominant envelope peak at {dom:.0f} Hz coincides with the inner-race rate."
    return (f"The dominant envelope peak at {dom:.0f} Hz coincides with neither "
            f"characteristic rate.")

def evidence_sentences(f):
    return [CHAR[_bin('env_char_energy_frac', f['env_char_energy_frac'])],
            OUTER_E(_bin('env_bpfo_ratio', f['env_bpfo_ratio']), f['bpfo_hz']),
            OUTER_H(_bin('env_bpfo_harmonic_ratio', f['env_bpfo_harmonic_ratio'])),
            INNER_E(_bin('env_bpfi_ratio', f['env_bpfi_ratio']), f['bpfi_hz']),
            INNER_S(_bin('env_bpfi_sideband_ratio', f['env_bpfi_sideband_ratio']),
                    f['f_shaft_hz']),
            dominant_peak_sentence(f)]

def build_explanation(true_label, f, rng):
    s = evidence_sentences(f)
    closing = {'Healthy':"Taken together, the signal evidence is consistent with a "
                         "healthy bearing.",
               'Inner_Ring_Fault':"Taken together, the signal evidence is consistent "
                         "with an inner ring fault.",
               'Outer_Ring_Fault':"Taken together, the signal evidence is consistent "
                         "with an outer ring fault."}[true_label]
    return ' '.join(s + [closing])

REQUIRED = {
 'Outer_Ring_Fault':[('env_bpfo_ratio','high'),('env_bpfo_harmonic_ratio','high')],
 'Inner_Ring_Fault':[('env_bpfi_ratio','high'),('env_bpfi_sideband_ratio','high')],
 'Healthy':[('env_char_energy_frac','low')],
}
PHRASE = {
 'env_bpfo_ratio'          : lambda f: f"concentrated energy at the outer-race rate near {f['bpfo_hz']:.1f} Hz",
 'env_bpfo_harmonic_ratio' : lambda f: "harmonic banding at twice and three times that rate",
 'env_bpfi_ratio'          : lambda f: f"concentrated energy at the inner-race rate near {f['bpfi_hz']:.1f} Hz",
 'env_bpfi_sideband_ratio' : lambda f: "a shaft-rate sideband below that rate",
 'env_char_energy_frac'    : lambda f: "most envelope energy spread across the band rather than concentrated at the characteristic frequencies",
}

def missing_evidence(wrong, true_label, f):
    art = {'Healthy':'A healthy bearing','Inner_Ring_Fault':'An inner ring fault',
           'Outer_Ring_Fault':'An outer-race defect'}[wrong]
    absent=[k for k,want in REQUIRED[wrong] if _bin(k, f[k]) != want]
    if absent:
        parts=[PHRASE[k](f) for k in absent]
        joined=parts[0] if len(parts)==1 else parts[0]+" with "+parts[1]
        return f"{art} would show {joined}; the envelope does not."
    present=[k for k,want in REQUIRED[true_label] if _bin(k, f[k]) == want]
    if present:
        return (f"{art} is not the better account here: {PHRESENT[present[0]](f)} "
                f"is what the envelope shows.")
    return (f"{art} is not supported by the evidence above, which is weak at "
            f"both characteristic rates.")

PHRESENT = {
 'env_bpfo_ratio'          : lambda f: f"concentrated energy at the outer-race rate near {f['bpfo_hz']:.1f} Hz",
 'env_bpfo_harmonic_ratio' : lambda f: "harmonic banding above the outer-race rate",
 'env_bpfi_ratio'          : lambda f: f"concentrated energy at the inner-race rate near {f['bpfi_hz']:.1f} Hz",
 'env_bpfi_sideband_ratio' : lambda f: "a shaft-rate sideband below the inner-race rate",
 'env_char_energy_frac'    : lambda f: "envelope energy spread across the band rather than concentrated at the characteristic frequencies",
}

def build_contradiction_explanation(true_label, wrong_label, f, rng):
    assert wrong_label != true_label, 'contrastive sample must supply a WRONG label'
    base = build_explanation(true_label, f, rng)
    return (base + f" However, the CNN classified this as {ARTICLED[wrong_label]}. "
            + missing_evidence(wrong_label, true_label, f))

def build_blind_observation(f, rng):
    return ' '.join(evidence_sentences(f))

def user_prompt(f, supplied, modality):
    freq = (f"At the current shaft speed of {f['speed_rpm']:.0f} RPM, "
            f"BPFO={f['bpfo_hz']:.1f} Hz and BPFI={f['bpfi_hz']:.1f} Hz.")
    parts=[]
    if modality in ('both','audio'):
        parts.append("<audio>Listen to this bearing vibration signal.</audio>")
    if modality in ('both','stft'):
        parts.append(f"<image>STFT spectrogram (0-{STFT_MAX_FREQ_LABEL} Hz) of the same "
                     f"signal. {freq}</image>")
    else:
        parts.append(freq)
    body="\n".join(parts)
    if supplied is None:
        return body + "\n\nDescribe what you observe in the audio and spectrogram."
    return (body + f"\n\nCNN classification: {PRETTY[supplied]}\n"
            "Examine the signal evidence and explain this diagnosis.")

_r=random.Random(0)
_demo=all_train[0]
_wrong=_r.choice([x for x in CLASSES if x!=_demo['label']])
print(f"--- prompt (contrastive: true={_demo['label']}, CNN says {_wrong}) ---")
print(user_prompt(_demo['features'],_wrong,MODALITY))
print('\n--- target ---')
print('<explanation>'+build_contradiction_explanation(
      _demo['label'],_wrong,_demo['features'],_r)+'</explanation>')

In [ ]:
#@title Checks on prompts and targets
DIAG=['env_bpfo_ratio','env_bpfi_ratio','env_char_energy_frac','env_dominant_freq_hz',
      'env_bpfo_harmonic_ratio','env_bpfi_sideband_ratio','kurtosis','crest_factor','rms']
man=manifest
probe=man['train'][0]['features']
leaks=[]
for mod in ('both','audio','stft'):
    for sup in [None]+CLASSES:
        p=user_prompt(probe,sup,mod)
        for k in DIAG:
            if k in ('bpfo_hz','bpfi_hz'): continue
            tok=f"{probe[k]:.3f}"[:5]
            if len(tok)>3 and tok in p: leaks.append((mod,sup,k))
assert not leaks, f'PROMPT LEAKS: {sorted(set(leaks))}'
print('leakage check passed')

BAD=['stationary defect','defect that passes','indicating a defect']
viol=[t for r in man['train'] if r['label']=='Healthy'
      for t in [build_explanation('Healthy', r['features'], random.Random(2))]
      if any(b in t for b in BAD)]
assert not viol, f'{len(viol)} Healthy targets assert a defect mechanism'
print('coherence check passed\n')

rr=random.Random(1)
by={}
for r in man['train']:
    by.setdefault(r['label'],[]).append(build_explanation(r['label'],r['features'],rr))
print()
for cls in CLASSES:
    v=by[cls]; u=len(set(v))
    print(f'{cls:18} n={len(v):5}  unique explanations={u:5}  ({100*u/len(v):.1f}%)')
for cls in CLASSES:
    print(f'\n--- {cls} example ---'); print(by[cls][0][:330]+'...')

In [ ]:
#@title Training set
train_dir=DATA_DIR/'train'; test_dir=DATA_DIR/'test'
def paths(split, rec):
    d = train_dir if split=='train' else test_dir
    return str(d/rec['audio']), rec['image']

rng=random.Random(SEED)
recs=list(man['train'])
if BALANCE_TRAIN:
    byc={}
    for r in recs: byc.setdefault(r['label'],[]).append(r)
    counts={k:len(v) for k,v in byc.items()}
    if BALANCE_MODE=='oversample':
        n_max=max(counts.values())
        recs=[]
        for k,v in byc.items():
            v=list(v); rng.shuffle(v)
            reps=[v[i % len(v)] for i in range(n_max)]
            recs+=reps
        rng.shuffle(recs)
        dup=sum(n_max-c for c in counts.values())
        print(f'oversampled to {n_max}/class -> {len(recs)}  '
              f'({dup} duplicated rows from {counts})')
    else:
        n_min=min(counts.values())
        for v in byc.values(): rng.shuffle(v)
        recs=[r for v in byc.values() for r in v[:n_min]]; rng.shuffle(recs)
        print(f'subsampled to {n_min}/class -> {len(recs)}')

n=len(recs); n_blind=int(round(n*BLIND_RATIO))
n_contr=int(round((n-n_blind)*CONTRASTIVE_RATIO/100.0))
idx=list(range(n)); rng.shuffle(idx)
blind_idx=set(idx[:n_blind]); contr_idx=set(idx[n_blind:n_blind+n_contr])

rows=[]
for i,r in enumerate(recs):
    f=r['features']; lab=r['label']; ap,ip=paths('train',r)
    if i in blind_idx:
        prompt=user_prompt(f,None,MODALITY); expl=build_blind_observation(f,rng)
        kind,sup='blind',None
    elif i in contr_idx:
        wrong=rng.choice([x for x in CLASSES if x!=lab])
        prompt=user_prompt(f,wrong,MODALITY)
        expl=build_contradiction_explanation(lab,wrong,f,rng); kind,sup='contrastive',wrong
    else:
        prompt=user_prompt(f,lab,MODALITY)
        expl=build_explanation(lab,f,rng); kind,sup='confirm',lab
    row={'messages':[{'role':'system','content':SYSTEM_PROMPT},
                     {'role':'user','content':prompt},
                     {'role':'assistant','content':f'<explanation>{expl}</explanation>'}],
         'meta':{'bearing':r['bearing'],'label':lab,'supplied':sup,'kind':kind}}
    if MODALITY in ('both','audio'): row['audios']=[ap]
    if MODALITY in ('both','stft'):  row['images']=[ip]
    rows.append(row)

train_jsonl=JSONL_DIR/f'train_{RUN_TAG}_{TEMPLATE_ID}.jsonl'
with open(train_jsonl,'w') as fh:
    for row in rows: fh.write(json.dumps({k:v for k,v in row.items() if k!='meta'})+'\n')
print(f'wrote {train_jsonl.name}  n={len(rows)}')
print('kind :', dict(Counter(r['meta']['kind'] for r in rows)))
print('label:', dict(Counter(r['meta']['label'] for r in rows)))
nc=sum(r['meta']['kind']=='contrastive' for r in rows)
nb_=sum(r['meta']['kind']=='blind' for r in rows)
print(f'contrastive {100*nc/(len(rows)-nb_):.1f}% of labelled rows | blind {100*nb_/len(rows):.1f}%')

In [ ]:
#@title Evaluation sets
test_recs=list(man['test'])
CONTRASTIVE_MAP={'Healthy':'Outer_Ring_Fault','Inner_Ring_Fault':'Outer_Ring_Fault',
                 'Outer_Ring_Fault':'Inner_Ring_Fault'}
def erow(r,sup,kind):
    f=r['features']; ap,ip=paths('test',r)
    row={'messages':[{'role':'system','content':SYSTEM_PROMPT},
                     {'role':'user','content':user_prompt(f,sup,MODALITY)}],
         'meta':{'bearing':r['bearing'],'label':r['label'],'supplied':sup,'kind':kind,
                 'features':{k:f[k] for k in
                    ['bpfo_hz','bpfi_hz','f_shaft_hz','env_bpfo_ratio','env_bpfi_ratio',
                     'env_bpfo_harmonic_ratio','env_bpfi_sideband_ratio',
                     'env_char_energy_frac','env_dominant_freq_hz']}}}
    if MODALITY in ('both','audio'): row['audios']=[ap]
    if MODALITY in ('both','stft'):  row['images']=[ip]
    return row

sets={'correct'    :[erow(r,r['label'],'confirm') for r in test_recs],
      'contrastive':[erow(r,CONTRASTIVE_MAP[r['label']],'reject') for r in test_recs],
      'blind'      :[erow(r,None,'blind') for r in test_recs]}
for nm,rws in sets.items():
    p=JSONL_DIR/f'eval_{nm}_{DATASET}_{MODALITY}_{TEMPLATE_ID}.jsonl'
    if p.exists():
        _old=[json.loads(l) for l in open(p) if l.strip()]
        assert _old==json.loads(json.dumps(rws)), f'{p.name} differs from the set built here'
        print(f'{nm:12} {len(rws):5} == {p.name} (matches, not rewritten)')
    else:
        with open(p,'w') as fh:
            for row in rws: fh.write(json.dumps(row)+'\n')
        print(f'{nm:12} {len(rws):5} -> {p.name}')

In [ ]:
#@title Run record
import math
n_train=len(rows)
steps_per_epoch=math.ceil(n_train/(BATCH_SIZE*GRAD_ACCUM))
planned_steps=steps_per_epoch*NUM_EPOCHS
OUT_DIR=CKPT_ROOT/f'{RUN_TAG}_{TEMPLATE_ID}'
_rec=rec_load(RUN_TAG)
ALREADY_TRAINED=bool(_rec and _rec.get('status')=='done' and _rec.get('ckpt')
                     and Path(_rec['ckpt']).exists() and _rec.get('steps_actual')==planned_steps)
if ALREADY_TRAINED:
    print(f'{RUN_TAG} is already trained ({_rec["steps_actual"]} steps, {_rec["ckpt"]}).')
else:
    rec_update(RUN_TAG, dataset=DATASET, model=MODEL_TAG, model_id=MODEL_ID, modality=MODALITY,
               ratio=CONTRASTIVE_RATIO, seed=SEED, spec=SPEC_ID, template=TEMPLATE_ID,
               variant=VARIANT, n_train=n_train, planned_steps=planned_steps, epochs=NUM_EPOCHS,
               lr=LR, lora_rank=LORA_RANK, lora_alpha=LORA_ALPHA, balanced=BALANCE_TRAIN,
               balance_mode=BALANCE_MODE, blind_ratio=BLIND_RATIO, train_jsonl=str(train_jsonl),
               out_dir=str(OUT_DIR), policy='final checkpoint, no selection', status='running', ckpt=None)
    print('recorded', RUN_TAG)
print(f'n_train {n_train} | steps/epoch {steps_per_epoch} | planned {planned_steps}')

In [ ]:
#@title Train
import os, glob, importlib.util
os.environ['MODELSCOPE_CACHE']=str(MODEL_CACHE)
OUT_DIR.mkdir(parents=True, exist_ok=True)
if importlib.util.find_spec('bitsandbytes') is None:
    get_ipython().system('pip install -q -U bitsandbytes')

def find_ckpts(root):
    cs=glob.glob(str(root/'**'/'checkpoint-*'),recursive=True)
    cs=[x for x in cs if os.path.isdir(x) and x.rsplit('-',1)[1].isdigit()]
    return sorted(cs,key=lambda p:int(p.rsplit('-',1)[1]))
ckpts=find_ckpts(OUT_DIR); resume=ckpts[-1] if ckpts else None
print('checkpoints:',len(ckpts),'| resuming from:',resume or '(fresh start)')

cmd=f'''TQDM_MININTERVAL=30 MODELSCOPE_CACHE={MODEL_CACHE} swift sft \
  --model {MODEL_ID} \
  --tuner_type lora \
  --lora_rank {LORA_RANK} --lora_alpha {LORA_ALPHA} --target_modules all-linear \
  --dataset {train_jsonl} \
  --num_train_epochs {NUM_EPOCHS} \
  --per_device_train_batch_size {BATCH_SIZE} \
  --gradient_accumulation_steps {GRAD_ACCUM} \
  --learning_rate {LR} \
  --max_length {MAX_LENGTH} \
  --torch_dtype bfloat16 \
  --optim adamw_8bit \
  --save_steps {SAVE_STEPS} --save_total_limit {SAVE_LIMIT} \
  --logging_steps 5 \
  --output_dir {OUT_DIR} \
  --dataloader_num_workers 2 \
  --seed {SEED}'''
if resume: cmd+=f' \\\n  --resume_from_checkpoint {resume}'
print(cmd)
if ALREADY_TRAINED:
    print('\nalready trained, skipping')
else:
    get_ipython().system(cmd)

In [ ]:
#@title Mark done
import glob, os
def find_ckpts(root):
    cs=glob.glob(str(root/'**'/'checkpoint-*'),recursive=True)
    cs=[x for x in cs if os.path.isdir(x) and x.rsplit('-',1)[1].isdigit()]
    return sorted(cs,key=lambda p:int(p.rsplit('-',1)[1]))
ckpts=find_ckpts(OUT_DIR); assert ckpts,'no checkpoint found'
final=ckpts[-1]; steps=int(final.rsplit('-',1)[1])
print('final:',final); print(f'steps {steps} of {planned_steps}')
if ALREADY_TRAINED:
    print('already marked done')
elif steps<planned_steps:
    print('incomplete: run the training cell again to resume')
else:
    rec_update(RUN_TAG,status='done',ckpt=final,steps_actual=steps)
    print('marked done.')

In [ ]:
#@title Evaluation configuration
CONDITIONS=['correct','contrastive','blind']
PER_CLASS=None
SUBSET_SEED=12345
RUN_SUBSTITUTION=True; SUBST_SEED=999
MAX_NEW_TOKENS=320
INFER_EXTRA='--max_batch_size 8'
ATTN_IMPL=None
RUN_ID=RUN_TAG
rec=rec_load(RUN_ID)
assert rec, f'{RUN_ID} not recorded: run the training cells first'
assert rec.get('status')=='done', f"status {rec.get('status')}: training incomplete"
CKPT=rec['ckpt']; assert Path(CKPT).exists(), f'checkpoint missing: {CKPT}'
CKPT_TAG=Path(CKPT).name
SPEC=json.load(open(DATA_DIR/'spec_cw_v2.json')); TH=SPEC['thresholds']
manifest=json.load(open(DATA_DIR/'data_manifest_cw_v2.json'))
print('RUN_ID', RUN_ID); print('ckpt  ', CKPT)
print('steps ', rec.get('steps_actual'), 'of', rec.get('planned_steps'))
print('evaluation on the whole test set:', len(manifest['test']), 'segments',
      dict(Counter(s['label'] for s in manifest['test'])))

In [ ]:
#@title Evaluation subsets
import random
from collections import Counter
def load_jsonl(p):
    with open(p) as f: return [json.loads(l) for l in f if l.strip()]

CONDITIONS=['correct','contrastive','blind']
subset_paths={}; infer_paths={}
for cond in CONDITIONS:
    src=JSONL_DIR/f'eval_{cond}_{DATASET}_{MODALITY}_{TEMPLATE_ID}.jsonl'
    assert src.exists(), f'missing {src.name}'
    dst=SUBSET_DIR/f'sub_{cond}_{DATASET}_{MODALITY}_all.jsonl'
    rows=load_jsonl(src)
    if PER_CLASS is None:
        with open(dst,'w') as f:
            for r in rows: f.write(json.dumps(r)+'\n')
        print(f'{cond:12} all {len(rows)} rows  '
              f'{dict(Counter(r["meta"]["label"] for r in rows))}')
    else:
        by={}
        for r in rows: by.setdefault(r['meta']['label'],[]).append(r)
        rng=random.Random(SUBSET_SEED); out=[]
        for cls in sorted(by):
            pool=sorted(by[cls], key=lambda r: r['meta']['bearing']
                        + r['messages'][1]['content'][:40])
            rng.shuffle(pool); out+=pool[:PER_CLASS]
        rng.shuffle(out)
        with open(dst,'w') as f:
            for r in out: f.write(json.dumps(r)+'\n')
        print(f'{cond:12} {len(out)} rows at {PER_CLASS}/class')
    subset_paths[cond]=dst
    q=dst.with_name(dst.stem+'_infer.jsonl')
    with open(q,'w') as f:
        for r in load_jsonl(dst):
            f.write(json.dumps({k:v for k,v in r.items() if k!='meta'})+'\n')
    infer_paths[cond]=q

if RUN_SUBSTITUTION:
    import random as _rnd
    base=load_jsonl(subset_paths['blind'])
    n=len(base); perm=list(range(n)); _r=_rnd.Random(SUBST_SEED); _r.shuffle(perm)
    for i in range(n):
        if perm[i]==i:
            k=(i+1)%n; perm[i],perm[k]=perm[k],perm[i]
    assert all(perm[i]!=i for i in range(n)), 'not a derangement'
    dst=SUBSET_DIR/f'sub_substituted_{DATASET}_{MODALITY}_all.jsonl'
    with open(dst,'w') as f:
        for i,r in enumerate(base):
            q=json.loads(json.dumps(r))
            src_row=base[perm[i]]
            if 'audios' in q: q['audios']=src_row['audios']
            if 'images' in q: q['images']=src_row['images']
            f.write(json.dumps(q)+'\n')
    subset_paths['substituted']=dst
    q=dst.with_name(dst.stem+'_infer.jsonl')
    with open(q,'w') as f:
        for r in load_jsonl(dst):
            f.write(json.dumps({k:v for k,v in r.items() if k!='meta'})+'\n')
    infer_paths['substituted']=q
    CONDITIONS=CONDITIONS+['substituted']
    print(f'substituted  built n={n} (media deranged, text and labels unchanged)')
rec_update(RUN_ID, subsets={c:str(p) for c,p in subset_paths.items()})

In [ ]:
#@title Generate
import time, shutil
RESULT_DIR=EVAL_DIR/f'{RUN_ID}_{TEMPLATE_ID}_{CKPT_TAG}'; RESULT_DIR.mkdir(parents=True,exist_ok=True)
print('results ->',RESULT_DIR)
def n_lines(p): return sum(1 for l in open(p) if l.strip()) if p.exists() else 0
result_paths={}
for cond in CONDITIONS:
    out=RESULT_DIR/f'infer_{cond}.jsonl'; result_paths[cond]=out
    need=n_lines(infer_paths[cond]); have=n_lines(out)
    if have==need:
        print(f'{cond}: complete ({have}/{need}), skipping'); continue
    if have:
        bak=out.with_suffix(f'.partial_{int(time.time())}.jsonl'); shutil.move(out,bak)
        print(f'{cond}: partial ({have}/{need}) moved to {bak.name}; regenerating')
    print(f'\n===== {cond} ({need} rows) ====='); t0=time.time()
    cmd=(f'TQDM_MININTERVAL=30 MODELSCOPE_CACHE={MODEL_CACHE} swift infer '
         f'--adapters {CKPT} --val_dataset {infer_paths[cond]} --result_path {out} '
         f'--max_new_tokens {MAX_NEW_TOKENS} --temperature 0 '
         f'--infer_backend transformers {INFER_EXTRA}'
         + (f' --attn_impl {ATTN_IMPL}' if ATTN_IMPL else ''))
    print(cmd); rc=os.system(cmd)
    have=n_lines(out)
    print(f'{cond}: exit {rc}, {have}/{need} rows, {(time.time()-t0)/60:.1f} min')
    if have!=need:
        raise RuntimeError(f'{cond}: swift infer wrote {have}/{need} rows (exit {rc})')

print('\nall conditions complete:')
for cond in CONDITIONS: print(f'   {cond:12} {n_lines(result_paths[cond])}/{n_lines(infer_paths[cond])}')
rec_update(RUN_ID, result_dir=str(RESULT_DIR), conditions=CONDITIONS)